# Chess - full pipeline, one session (stages 02 -> 10)

Branch `perf/batched-mcts` of `TahaSalmani/Chess`, including the Pre-LN fix for the policy-net collapse.

**Before anything else: Runtime -> Change runtime type -> Hardware accelerator: GPU (T4).** The first code cell asserts it and stops the run if you forgot.

Then **Runtime -> Run all** is enough; nothing below needs input from you.

| phase | what | time |
|---|---|---|
| setup | GPU check, clone, deps | ~2 min |
| data | 6000 Lichess games -> ~250k positions -> tensors | ~4 min |
| train | stage 09, 50 epochs, 10 layers | **~1.5-2.5 h** |
| gate | top-1 accuracy + collapse check, writes `rl_gate.txt` | ~1 min |
| RL | stage 10 self-play + one RL step, **skipped automatically if the gate fails** | ~10-20 min |
| save | download every artifact | ~1 min |

Two things worth knowing:

- **Colab kills idle sessions.** Keep the tab open. The stage 09 training run is the long one and there is no way to resume it.
- **RAM ceiling is ~13 GB.** Stage 07 materialises the whole train split as float32, ~3.1 GB at 250k positions. Do not raise `NUM_GAMES_TO_FETCH` past ~12000 in this notebook.
- All tunables live in the **first code cell** (`NUM_GAMES_TO_FETCH`, `MIN_TOP1_FOR_RL`, `RL_PARAMS`). Edit that one cell, then Run all.

The gate exists because stage 10 is pointless on a collapsed policy net: MCTS would search with a chance-level prior and every self-play value target comes out 0.0, so the value head just learns to output zero. That already happened once.

In [ ]:
# ---- run-all config -------------------------------------------------------
NUM_GAMES_TO_FETCH = 6000     # ~41 positions/game -> ~250k positions. RAM-bound, see above.
MIN_TOP1_FOR_RL    = 0.10     # below this the MCTS prior is ~chance level; stage 10 is skipped

# stage 10 overrides (patched into params.yaml just before stage 10 runs). The repo defaults are
# sized for a local CPU smoke test and are wrong for a real run:
#   MAX_MOVES 50  -> every game hits the cap -> every value target is 0.0
#   EPOCHS 50     -> shared with stage 09; 50 epochs on a few thousand positions overfits
RL_PARAMS = {
    "NUM_SIMULATION": 100,
    "NUM_GAMES": 32,
    "MAX_MOVES": 200,
    "POOL_SIZE": 32,
    "RL_EPOCHS": 10,
}
print("config:", NUM_GAMES_TO_FETCH, "games | gate at", f"{MIN_TOP1_FOR_RL:.0%}", "| RL", RL_PARAMS)

In [ ]:
import os, subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout.strip())
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
cores = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else os.cpu_count()
print('usable vCPU:', cores)
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> GPU (T4), then re-run all'

In [ ]:
!rm -rf /content/Chess
!git clone --branch perf/batched-mcts --single-branch --depth 1 https://github.com/TahaSalmani/Chess.git /content/Chess
%cd /content/Chess
!pip install -q python-chess zstandard scikit-learn tensorboard python-box==6.0.2 ensure pyyaml
!pip install -q -e .
!git log --oneline -1

## Phase 1 - data

`research/fetch_pgn_sample.py` issues an HTTP byte-range request, so it pulls a few MB instead of the 3.5 GB monthly database. Stage 01 is skipped on purpose: it downloads the whole file.

Stage 03 keeps only games where both players are 1200+ and drops time forfeits, then emits one `(board, move)` pair per ply with the move encoded as `from_square * 64 + to_square` into 4096 classes.

In [ ]:
%%time
!python research/fetch_pgn_sample.py {NUM_GAMES_TO_FETCH}
!ls -la artifacts/Data/

In [ ]:
%%time
!python src/Chess/pipeline/stage_02_DataValidation.py 2>&1 | grep -vE 'Created directory|Config loaded'
!python src/Chess/pipeline/stage_03_DataTransformation.py 2>&1 | grep -vE 'Created directory|Config loaded'
!ls -la artifacts/data_transformation/

In [ ]:
%%time
!python src/Chess/pipeline/stage_07_TorchDataset.py 2>&1 | grep -vE 'Created directory|Config loaded'
!python src/Chess/pipeline/stage_08_TorchBaseModel.py 2>&1 | grep -vE 'Created directory|Config loaded'
!ls -la artifacts/torch_dataset artifacts/torch_base_model
!cat params.yaml

## Phase 2 - stage 09, supervised policy training

This is the long cell. The Chess logger prints a line every 10 steps (~620 lines per epoch), so the grep below keeps only the per-epoch summary, every 1000th step, checkpoint saves and early-stopping messages. That is roughly one progress line every 20-30 seconds. The unfiltered log is still written to `stage09_full.log`.

`EarlyStopping(patience=5)` may cut the run short - intended, and `best_model.pth` keeps the best epoch seen.

Training now uses `OneCycleLR` warmup over the first 10% of steps, so **the loss barely moves for the first epoch or two.** That is the warmup ramp, not a stall. The unfiltered log is still on disk as `stage09_full.log` if you want every step.

In [ ]:
%%time
!stdbuf -oL python -u src/Chess/pipeline/stage_09_Train_Torch_Model.py 2>&1 \
  | tee stage09_full.log \
  | grep --line-buffered -E 'device|Epoch \[[0-9]+/[0-9]+\] - |Step \[[0-9]*000/|Checkpoint saved|EarlyStopping|Early stopping|Final trained'

## Phase 3 - the gate

Cross-entropy over 4096 classes is hard to read on its own, so these are the reference points:

| what it means | loss | top-1 |
|---|---|---|
| uniform guess over 4096 classes | 8.32 | 0.02% |
| always predict the single most common move | 6.61 | 1.26% |
| clearly learning | < 5.0 | > 10% |
| strong policy net on human games | ~2.5 | 40-55% |

The `distinct moves predicted` count is the real collapse detector: 1 means the net ignores the position entirely. A previous run scored 1.20% top-1 after 50 epochs with exactly that signature.

Realistic target at ~250k positions with this mean-pooled head is **10-25% top-1**, not 40-55%.

This cell writes `rl_gate.txt`; the stage 10 cell reads it and skips training on a fail, so **Run all is safe**.

In [ ]:
import sys, numpy as np, torch
sys.path.insert(0, '/content/Chess/src')
from Chess.components.torch_base_model import ChessPolicyNet
from Chess.config.configuration import ConfigurationManager

cm = ConfigurationManager()
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ChessPolicyNet(config=cm.get_torch_base_model()).to(dev)
model.load_state_dict(torch.load('artifacts/prepare_torch_callbacks/best_model.pth', map_location=dev))
model.eval()

X = np.load('artifacts/data_transformation/X_test.npy')
y = np.load('artifacts/data_transformation/y_test.npy').astype(np.int64)
print(f'test positions: {len(y)}')

correct = top5 = 0
preds = []
with torch.no_grad():
    for i in range(0, len(y), 1024):
        xb = torch.from_numpy(X[i:i+1024]).float().to(dev)
        logits = model(xb)
        pred = logits.argmax(1).cpu().numpy()
        preds.append(pred)
        correct += int((pred == y[i:i+1024]).sum())
        top5 += int((logits.topk(5, dim=1).indices.cpu().numpy() == y[i:i+1024, None]).any(1).sum())

preds = np.concatenate(preds)
uniq, counts = np.unique(preds, return_counts=True)
top1 = correct / len(y)
top_share = counts.max() / len(preds)

print(f'top-1 accuracy: {top1 * 100:.2f}%   (random 0.02%, marginal 1.26%)')
print(f'top-5 accuracy: {top5 / len(y) * 100:.2f}%')
print(f'distinct moves predicted: {len(uniq)} of 4096')
print(f'share of most-predicted move: {top_share * 100:.2f}%')

if top_share > 0.5:
    verdict = 'COLLAPSED'
    print('\nCOLLAPSED: one fixed move regardless of position. Do not run stage 10.')
elif top1 < MIN_TOP1_FOR_RL:
    verdict = 'TOO_WEAK'
    print(f'\nTOO WEAK: {top1:.2%} < {MIN_TOP1_FOR_RL:.0%}. The MCTS prior would be near chance level.')
else:
    verdict = 'PASS'
    print('\nPASS: stage 10 will run.')

with open('rl_gate.txt', 'w') as f:
    f.write(f'{verdict} top1={top1:.6f} top5={top5/len(y):.6f} distinct={len(uniq)} share={top_share:.6f}\n')
print('wrote rl_gate.txt ->', verdict)

## Phase 4 - stage 10, RL

Two overrides matter here and the repo defaults get both wrong:

- **`MAX_MOVES: 50`** - every self-play game hits the cap instead of finishing, so the result is scored 0.0 for all of them and every value target becomes exactly 0.0. The value head then learns to output zero and the value term in PUCT is dead. `200` lets games actually terminate.
- **`EPOCHS: 50`** is shared with stage 09. Running 50 epochs over a few thousand self-play positions overfits them and forgets the supervised policy. `RL_EPOCHS: 10` below is used instead.

`POOL_SIZE` games are searched in lockstep and share one batched forward per simulation, so `NUM_GAMES == POOL_SIZE` gives the best GPU utilisation.

The next cell reads `rl_gate.txt` and exits without training if the gate did not pass.

In [ ]:
import re, pathlib
p = pathlib.Path('params.yaml')
text = p.read_text()
before = dict(re.findall(r'^(\w+)\s*:\s*(.+?)\s*$', text, re.M))

for key, value in RL_PARAMS.items():
    target = 'EPOCHS' if key == 'RL_EPOCHS' else key
    if re.search(rf'^{target}\s*:', text, re.M):
        text = re.sub(rf'^{target}\s*:.*$', f'{target} : {value}', text, flags=re.M)
    else:
        text += f'\n{target} : {value}\n'
p.write_text(text)

after = dict(re.findall(r'^(\w+)\s*:\s*(.+?)\s*$', p.read_text(), re.M))
print('params.yaml changed for the RL phase:')
for k in sorted(set(before) | set(after)):
    if before.get(k) != after.get(k):
        print(f'  {k:16s} {before.get(k, "-"):>8s} -> {after.get(k, "-")}')

In [ ]:
%%time
gate = open('rl_gate.txt').read().split()[0]
if gate != 'PASS':
    print(f'SKIPPING stage 10: gate said {gate}. Fix supervised training first.')
else:
    print('gate PASS - running stage 10')
    !stdbuf -oL python -u src/Chess/pipeline/stage_10_Pytorch_RL.py 2>&1 | tee stage10_full.log | grep --line-buffered -E 'device|weights|MCTS config|Self-play|round |Game |Epoch |saved|completed|Error|Traceback'

## Phase 5 - did RL actually move the needle?

Stage 10 is one iteration, not a full AlphaZero loop, so the bar is low but real:

- **`Value` loss should not go to ~0.** If it does, every self-play target was 0.0 and the value head learned a constant - check the `result (White)` values in the stage 10 log. A spread of +1.0 / -1.0 / 0.0 means games terminated properly.
- **`Policy` loss should fall** from roughly the supervised cross-entropy toward the entropy of the MCTS visit distribution.
- **top-1 vs the MCTS target** is the honest measure: if the RL step improved agreement with search, it helped.

A degenerate run is not a failure of the code, it is a signal that `NUM_GAMES` or `NUM_SIMULATION` is too small for the net's current strength.

In [ ]:
import os, numpy as np, torch, sys
sys.path.insert(0, '/content/Chess/src')
from Chess.components.policy_value_torch_model import ValuePolicyNet
from Chess.config.configuration import ConfigurationManager

cm = ConfigurationManager()
rl = cm.get_rl_config()
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

d = str(rl.self_play_data)
rl_weights = str(rl.root_dir / 'rl_trained_model.pth')

def verify():
    if not os.path.exists(f'{d}/selfplay_x.pt') or not os.path.exists(rl_weights):
        print('stage 10 produced no output (the gate probably blocked it). Nothing to verify.')
        return

    sx = torch.load(f'{d}/selfplay_x.pt', map_location='cpu')
    sp = torch.load(f'{d}/selfplay_policy.pt', map_location='cpu')
    sv = torch.load(f'{d}/selfplay_value.pt', map_location='cpu')
    print(f'self-play positions: {len(sx)}')
    vals, cnts = np.unique(sv.numpy(), return_counts=True)
    print('value target distribution:', {float(v): int(c) for v, c in zip(vals, cnts)})
    if set(vals.tolist()) == {0.0}:
        print('  !! every value target is 0.0 - all games hit MAX_MOVES instead of finishing. '
              'Raise MAX_MOVES in RL_PARAMS and re-run stage 10.')
    print('policy target entropy (nats): %.3f'
          % float(-(sp.clamp_min(1e-12) * sp.clamp_min(1e-12).log()).sum(1).mean()))

    net = ValuePolicyNet(config=cm.get_torch_base_model()).to(dev)
    net.load_state_dict(torch.load(rl_weights, map_location=dev))
    net.eval()
    agree = 0
    with torch.no_grad():
        for i in range(0, len(sx), 512):
            logits, _ = net(sx[i:i+512].to(dev))
            agree += int((logits.argmax(1).cpu() == sp[i:i+512].argmax(1)).sum())
    print(f'agreement with the MCTS best move: {agree / len(sx) * 100:.2f}%')

verify()

In [ ]:
import os
from google.colab import files
for path in ['artifacts/prepare_torch_callbacks/best_model.pth',
             'artifacts/prepare_torch_callbacks/trained_model.pth',
             'artifacts/rl_model/rl_trained_model.pth',
             'rl_gate.txt', 'stage09_full.log', 'stage10_full.log']:
    if os.path.exists(path):
        print('downloading', path, f'({os.path.getsize(path)/1e6:.1f} MB)')
        files.download(path)
    else:
        print('missing  ', path)
print('\nbest_model.pth is the one to keep: trained_model.pth is whatever the last epoch left, '
      'which EarlyStopping places up to 5 epochs past the best.')

## Scaling up / troubleshooting

**More data is the biggest lever**, ahead of any architecture change. `NUM_GAMES_TO_FETCH = 25000` gives ~1M positions but stage 09 then runs 4-5 h and outlives a free Colab session, and stage 07 needs ~7 GB of RAM. Past that point:

- stream from the `.npy` files with a memmap dataset instead of materialising one float32 tensor
- Kaggle has 9 h sessions, 29 GB RAM and 30 free GPU-hours/week, but this account currently gets **no DNS at all inside kernels**, so git clone / pip install / the PGN download all fail. Kaggle needs phone verification to enable internet; without it the repo, the PGN and every missing wheel have to ship in as an attached Dataset.

**If the gate says COLLAPSED** (top-1 ~1.3%, `distinct moves predicted: 1`), the transformer stopped learning. That was the Post-LN failure mode and it is fixed on this branch, so if it comes back check that the `git log --oneline -1` output under the clone cell shows the Pre-LN commit.

**If the gate says TOO_WEAK** but distinct moves is in the hundreds, training is healthy and just needs more data or more epochs. Raise `NUM_GAMES_TO_FETCH` before touching the model.

**Recovering from a disconnect:** stage 09 writes `best_model.pth` every time validation loss improves, but Colab's disk is ephemeral, so a disconnect loses it. For a run you cannot babysit, mount Drive in a cell *before* the stage 09 cell and copy the checkpoint there afterwards:

```python
from google.colab import drive; drive.mount('/content/drive')
```

i.e.:

```python
!cp artifacts/prepare_torch_callbacks/best_model.pth /content/drive/MyDrive/
```

The mount needs an interactive auth code, which is why it is not part of Run all.